# Package

In [ ]:
# ============================================================
# 1) Core Python & Paths
# ============================================================
import sys
import os
import io
import json
import joblib
import warnings
import logging
from pathlib import Path
from tempfile import TemporaryDirectory
from contextlib import redirect_stdout, redirect_stderr
from dateutil.relativedelta import relativedelta

# Project root (notebook dans /notebooks)
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)


# ============================================================
# 2) Data Handling & Custom Utils (Feast + helpers)
# ============================================================
import pandas as pd
import numpy as np

from utils import load_wide_from_feast, build_unrate_exog_dataset
from experiment_utils import to_wide_from_oos, make_mae_dm_pivot


# ============================================================
# 3) Forecasting Models (MLForecast + ML)
# ============================================================
from mlforecast import MLForecast
from mlforecast.utils import PredictionIntervals

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV
from lightgbm import LGBMRegressor


# ============================================================
# 4) Evaluation & Statistical Tests
# ============================================================
from scipy.stats import ttest_rel
from statsmodels.stats.contingency_tables import mcnemar


# ============================================================
# 5) Visualization
# ============================================================
from utilsforecast.plotting import plot_series
from IPython.display import IFrame, display


# ============================================================
# 6) Experiment Tracking (MLflow)
# ============================================================
import mlflow
from mlflow.tracking import MlflowClient

# Importation des données

In [2]:
series_ids = [
    "BUSLOANS","CPIAUCSL","DPCERA3M086SBEA","INDPRO",
    "M2SL","OILPRICEX","RPI","SP500","TB3MS","UNRATE","USREC",
]

START = "1960-01-01"
END   = "2025-08-01"

In [3]:
# ============================================================
# 1) WIDE dataset (df) depuis Feast
# ============================================================
import pandas as pd

LAG = 6  # ✅ tout lag = 12 (aucune variable contemporaine)

df = load_wide_from_feast(
    "stationary_value:value",
    series_ids,
    start=START,
    end=END
)

# ============================================================
# 2) Convert WIDE -> LONG (obligatoire pour build_unrate_exog_dataset)
# ============================================================
df_stationary = (
    df.reset_index()
      .melt(id_vars="date", var_name="series_id", value_name="value")
)

# ============================================================
# 3) Build target + exog + MLForecast format
# ============================================================
df_model, ts_lr, exog_cols = build_unrate_exog_dataset(df_stationary)

# ============================================================
# 4) ✅ TOUTES les exog laggées de 12 mois (aucune contemporaine)
#    - on crée BUSLOANS_lag12, CPIAUCSL_lag12, ...
#    - on supprime BUSLOANS, CPIAUCSL, ... (contemporaines)
# ============================================================
ts_lr = ts_lr.sort_values(["unique_id", "ds"]).copy()

exog_cols_lag12 = []
for c in exog_cols:
    new_c = f"{c}_lag{LAG}"
    ts_lr[new_c] = ts_lr.groupby("unique_id")[c].shift(LAG)
    exog_cols_lag12.append(new_c)

# supprimer les exog contemporaines
ts_lr = ts_lr.drop(columns=exog_cols)

# mettre à jour la liste exog utilisée par MLForecast
exog_cols = exog_cols_lag12

# drop lignes où les lag12 n'existent pas (12 premiers mois)
ts_lr = ts_lr.dropna(subset=["y"] + exog_cols).reset_index(drop=True)

# ============================================================
# 5) Prints / checks
# ============================================================
print("df (wide) shape:", df.shape)
print("df_stationary (long) shape:", df_stationary.shape)
print("df_model shape:", df_model.shape)
print("ts_lr shape (after lag12):", ts_lr.shape)
print("Exog cols (lag12):", exog_cols)

print("\nPreview:")
print(ts_lr[["unique_id", "ds", "y"] + exog_cols[:5]].head(15))

Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
df (wide) shape: (788, 11)
df_stationary (long) shape: (8668, 3)
df_model shape: (788, 12)
ts_lr shape (after lag12): (782, 13)
Exog cols (lag12): ['BUSLOANS_lag6', 'CPIAUCSL_lag6', 'DPCERA3M086SBEA_lag6', 'INDPRO_lag6', 'M2SL_lag6', 'OILPRICEX_lag6', 'RPI_lag6', 'SP500_lag6', 'TB3MS_lag6', 'USREC_lag6']

Preview:
   unique_id         ds    y  BUSLOANS_lag6  CPIAUCSL_lag6  \
0     UNRATE 1960-07-01  0.4       0.011578      -0.006156   
1     UNRATE 1960-08-01  0.4       0.011905      -0.003767   
2     UNRATE 1960-09-01  0.0      -0.008356      -0.005455   
3     UNRATE 1960-10-01  0.4      -0.009098       0.005090   
4     UNRATE 1960-11-01  0.3      -0.000359       0.003383   
5     UNRATE 1960-12-01  1.3       0.014620       0.006777   
6     UNRATE 1961-01-01  1.4      -0.000611      -0.005433   
7     UNRATE 1961-02-01  2.1      -0.016888      -0.004074   
8     UNRATE 1961-03-01  1.

d:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\3_notebook\ML Experiment\utils.py:171: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("M")


# Model settings

In [4]:
# instancier le modèle
models = {"LR": LinearRegression(),
          "RIDGE" : Ridge (), 
          "LGBM" : LGBMRegressor()}

MLF_MODELS = {
    "LR_EXOG_ONLY": lambda freq: MLForecast(
        models=models,
        freq=freq,
        lags=[],               
        date_features=[],      
    )
}

In [5]:
# -----------------------------
# Paramètres
# -----------------------------
H = 12
STEP_SIZE = 1
PI_WINDOWS = 3
LEVELS = [95]
FREQ = "MS"

EXP_START = pd.Timestamp("1990-01-01")
EXP_END   = pd.Timestamp("2025-08-01")   # inclus

In [6]:
def _ensure_ms(x):
    x = pd.Timestamp(x)
    return x.to_period("M").to_timestamp(how="start").normalize()

def _n_windows_monthly(ds_start, ds_end):
    return (ds_end.year - ds_start.year) * 12 + (ds_end.month - ds_start.month) + 1

In [7]:
EXP_START = _ensure_ms(EXP_START)
EXP_END   = _ensure_ms(EXP_END)

# cutoff = ds - h mois
CUTOFF_START = EXP_START - relativedelta(months=H)
CUTOFF_END   = EXP_END   - relativedelta(months=H)
PARTITIONS   = _n_windows_monthly(CUTOFF_START, CUTOFF_END)

print("✅ EXP ds range          :", EXP_START.date(), "→", EXP_END.date())
print("✅ CUTOFF range          :", CUTOFF_START.date(), "→", CUTOFF_END.date())
print("✅ PARTITIONS (n_windows):", PARTITIONS)
print("ts_lr ds range           :", ts_lr["ds"].min().date(), "→", ts_lr["ds"].max().date())

# éviter fuite future (recommandé)
ts_lr = ts_lr[ts_lr["ds"] <= EXP_END].copy()

# -----------------------------
# Instancier le modèle
# -----------------------------
mlf = MLF_MODELS["LR_EXOG_ONLY"](FREQ)

model_names = list(mlf.models.keys())
first_name = next(iter(mlf.models))
print("Running models:", model_names)
print("First model class:", mlf.models[first_name].__class__.__name__)
print("Freq:", FREQ)

✅ EXP ds range          : 1990-01-01 → 2025-08-01
✅ CUTOFF range          : 1989-01-01 → 2024-08-01
✅ PARTITIONS (n_windows): 428
ts_lr ds range           : 1960-07-01 → 2025-08-01
Running models: ['LR', 'RIDGE', 'LGBM']
First model class: LinearRegression
Freq: MS


# Backtesting

In [8]:
# ============================================================
# Backtesting  (style "exercise")
# ============================================================
h = H
step_size = STEP_SIZE
partitions = PARTITIONS
n_windows = PI_WINDOWS
method = "conformal_distribution"
levels = LEVELS

pi = PredictionIntervals(
    h=h,
    n_windows=n_windows,
    method=method,
)

bkt_lr = mlf.cross_validation(
    df=ts_lr,
    h=h,
    step_size=step_size,          # ✅ cutoff mensuel
    n_windows=partitions,         # ✅ nb partitions auto
    prediction_intervals=pi,
    level=levels,
    fitted=True,
    static_features=[],
)

meta = {
    "h": h,
    "step_size": step_size,
    "exp_start": EXP_START,
    "exp_end": EXP_END,
    "cutoff_start": CUTOFF_START,
    "cutoff_end": CUTOFF_END,
    "partitions": partitions,
    "pi_windows": n_windows,
    "conformal_method": method,
    "levels": levels,
}

# -----------------------------
# Filtrer sur l’expérience (sécurité)
# -----------------------------
bkt_lr_eval = bkt_lr[(bkt_lr["ds"] >= EXP_START) & (bkt_lr["ds"] <= EXP_END)].copy()

# ============================================================
# ✅ 1 seule ligne par ds : garder le cutoff le plus récent
# ============================================================
bkt_lr_final = (
    bkt_lr_eval
    .sort_values(["unique_id", "ds", "cutoff"])
    .groupby(["unique_id", "ds"], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

# -----------------------------
# Checks
# -----------------------------
print("bkt_lr_eval rows         :", len(bkt_lr_eval))
print("bkt_lr_final rows        :", len(bkt_lr_final))
print("bkt_lr_final ds range    :", bkt_lr_final["ds"].min().date(), "→", bkt_lr_final["ds"].max().date())

dup = bkt_lr_final.duplicated(subset=["unique_id", "ds"]).sum()
print("duplicates (unique_id, ds):", dup)

bkt_lr_final.head()

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000097 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 864
[LightGBM] [Info] Number of data points in the train set: 307, number of used features: 10
[LightGBM] [Info] Start training from score 0.071661
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

,unique_id,ds,cutoff,y,LR,RIDGE,LGBM,LR-lo-95,LR-hi-95,RIDGE-lo-95,RIDGE-hi-95,LGBM-lo-95,LGBM-hi-95
0,UNRATE,1990-01-01,1989-12-01,0.0,0.293094,-0.284023,-0.006856,-0.075968,0.662156,-0.766392,0.198347,-0.227466,0.213755
1,UNRATE,1990-02-01,1990-01-01,0.1,0.022359,-0.310885,-0.135945,-0.432703,0.477422,-0.854670,0.232899,-0.520775,0.248886
2,UNRATE,1990-03-01,1990-02-01,0.2,0.003990,-0.348619,-0.149455,-0.797507,0.805487,-0.959538,0.262301,-1.503871,1.204960
3,UNRATE,1990-04-01,1990-03-01,0.2,-0.188218,-0.391875,-0.017799,-0.849793,0.473358,-0.984998,0.201247,-0.586013,0.550415
4,UNRATE,1990-05-01,1990-04-01,0.2,-0.144925,-0.397146,-0.116772,-0.715670,0.425821,-0.911522,0.117230,-0.661214,0.427669


# Transformer le Backtesting

In [9]:
bkt_score = bkt_lr_final.copy()
bkt_score["ds"] = pd.to_datetime(bkt_score["ds"], errors="coerce")

# enlever timezone si jamais
if pd.api.types.is_datetime64tz_dtype(bkt_score["ds"]):
    bkt_score["ds"] = bkt_score["ds"].dt.tz_convert(None)

bkt_score = bkt_score.dropna(subset=["ds"])
bkt_score = bkt_score[(bkt_score["ds"] >= EXP_START) & (bkt_score["ds"] <= EXP_END)].reset_index(drop=True)

bins = pd.to_datetime(["1990-01-01","2000-01-01","2009-01-01","2020-01-01","2025-09-01"])
labels = ["1990-1999", "2000-2008", "2009-2019", "2020-end"]

bkt_score["partition"] = pd.cut(bkt_score["ds"], bins=bins, labels=labels, right=False, include_lowest=True)
bkt_score = bkt_score.dropna(subset=["partition"]).reset_index(drop=True)

print(bkt_score[["ds","cutoff","partition"]].head(10))
print(bkt_score["partition"].value_counts().sort_index())

          ds     cutoff  partition
0 1990-01-01 1989-12-01  1990-1999
1 1990-02-01 1990-01-01  1990-1999
2 1990-03-01 1990-02-01  1990-1999
3 1990-04-01 1990-03-01  1990-1999
4 1990-05-01 1990-04-01  1990-1999
5 1990-06-01 1990-05-01  1990-1999
6 1990-07-01 1990-06-01  1990-1999
7 1990-08-01 1990-07-01  1990-1999
8 1990-09-01 1990-08-01  1990-1999
9 1990-10-01 1990-09-01  1990-1999
partition
1990-1999    120
2000-2008    108
2009-2019    132
2020-end      68
Name: count, dtype: int64


C:\Users\Mita\AppData\Local\Temp\ipykernel_16588\1843901458.py:5: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(bkt_score["ds"]):


# Building Leaderbord

In [15]:
import numpy as np
import pandas as pd

tmp = bkt_score.copy()

models = ["LR"]

# 1) S'assurer que lower <= upper (sécurité)
for m in models:
    lo = f"{m}-lo-95"
    hi = f"{m}-hi-95"
    tmp[[lo, hi]] = np.sort(tmp[[lo, hi]].to_numpy(), axis=1)

# 2) Wide -> Long + features de scoring
rows = []
base_cols = ["unique_id", "ds", "cutoff", "y", "partition"]

for m in models:
    lo = f"{m}-lo-95"
    hi = f"{m}-hi-95"

    s = tmp[base_cols].copy()
    s["model_label"] = m
    s["model_name"]  = m

    s["forecast"] = tmp[m]
    s["lower"]    = tmp[lo]
    s["upper"]    = tmp[hi]

    s["abs_err"]   = (s["y"] - s["forecast"]).abs()
    s["covered"]   = ((s["y"] >= s["lower"]) & (s["y"] <= s["upper"])).astype(int)
    s["int_width"] = (s["upper"] - s["lower"]).abs()

    rows.append(s)

long_sc = pd.concat(rows, ignore_index=True)

# 3) Sanity + types
long_sc = long_sc.loc[:, ~long_sc.columns.duplicated()].copy()
long_sc["partition"] = long_sc["partition"].astype(str)

# 4) Ajouter la partition "ALL" (toutes partitions confondues) SANS DUPLICATION
#    -> on duplique uniquement la colonne "partition" en mettant "ALL"
long_all = long_sc.copy()
long_all["partition"] = "ALL"

long_sc2 = pd.concat([long_sc, long_all], ignore_index=True)

# 5) Score MAE / Coverage / Width / N par modèle et partition (incluant ALL)
score_df = (
    long_sc2
    .groupby(["unique_id", "model_label", "model_name", "partition"], observed=True)
    .agg(
        mae=("abs_err", "mean"),
        coverage=("covered", "mean"),
        width=("int_width", "mean"),
        n=("y", "size"),
    )
    .reset_index()
)

# 6) Top 3 par partition (inclut ALL)
leaderboard = (
    score_df.sort_values(
        by=["partition", "mae", "coverage", "width"],
        ascending=[True, True, False, True],
    )
    .groupby("partition", as_index=False)
    .head(2)
)

print("=== SCORE (MAE / Coverage / Width / N) ===")
print(score_df.sort_values(["partition", "mae"]).head(50))

print("\n=== TOP 2 par partition (incluant ALL) ===")
print(leaderboard)

=== SCORE (MAE / Coverage / Width / N) ===
  unique_id model_label model_name  partition       mae  coverage     width  \
0    UNRATE          LR         LR  1990-1999  0.421220  0.800000  1.419842   
1    UNRATE          LR         LR  2000-2008  0.377789  0.648148  0.981396   
2    UNRATE          LR         LR  2009-2019  0.597483  0.780303  1.902029   
3    UNRATE          LR         LR   2020-end  1.776643  0.764706  6.250181   
4    UNRATE          LR         LR        ALL  0.679970  0.750000  2.225355   

     n  
0  120  
1  108  
2  132  
3   68  
4  428  

=== TOP 2 par partition (incluant ALL) ===
  unique_id model_label model_name  partition       mae  coverage     width  \
0    UNRATE          LR         LR  1990-1999  0.421220  0.800000  1.419842   
1    UNRATE          LR         LR  2000-2008  0.377789  0.648148  0.981396   
2    UNRATE          LR         LR  2009-2019  0.597483  0.780303  1.902029   
3    UNRATE          LR         LR   2020-end  1.776643  0.764706  6

# Add AR on leaderbord

## Load AR from MLFLOW

In [28]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

EXP_NAME = "Baseline"
exp = mlflow.get_experiment_by_name(EXP_NAME)
assert exp is not None, f"Experiment introuvable: {EXP_NAME}"

runs = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="attributes.run_name LIKE 'AR_%'",
    output_format="pandas"
)

client = MlflowClient()

bkt_frames = []

for _, row in runs.iterrows():

    run_id = row["run_id"]

    try:
        artifacts = client.list_artifacts(run_id, path="data")

        for art in artifacts:
            if art.path.endswith(".parquet"):

                local_path = client.download_artifacts(run_id, art.path)
                df_tmp = pd.read_parquet(local_path)

                # ajouter infos run
                df_tmp["run_id"] = run_id
                df_tmp["partition_run"] = row.get("params.partition", None)

                bkt_frames.append(df_tmp)

    except Exception:
        continue

if len(bkt_frames) > 0:
    bkt_all = pd.concat(bkt_frames, ignore_index=True)
else:
    bkt_all = pd.DataFrame()

print("bkt_all shape:", bkt_all.shape)
bkt_all.head()

bkt_all shape: (2568, 9)


,ds,y_true,y_hat,lo_95,hi_95,partition,p_selected,run_id,partition_run
0,1990-01-01,0.0,0.038995,-0.896105,0.974096,1990-1999,4,4a85f69b86244df091ad640c3a96d467,ALL
1,1990-02-01,0.1,-0.158664,-0.963583,0.646255,1990-1999,4,4a85f69b86244df091ad640c3a96d467,ALL
2,1990-03-01,0.2,-0.338061,-1.205755,0.529634,1990-1999,4,4a85f69b86244df091ad640c3a96d467,ALL
3,1990-04-01,0.2,0.079964,-0.698902,0.858830,1990-1999,4,4a85f69b86244df091ad640c3a96d467,ALL
4,1990-05-01,0.2,-0.004353,-0.955797,0.947091,1990-1999,4,4a85f69b86244df091ad640c3a96d467,ALL


## Leaderbord from AR

In [29]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

EXP_NAME = "Baseline"
exp = mlflow.get_experiment_by_name(EXP_NAME)
assert exp is not None, f"Experiment introuvable: {EXP_NAME}"

runs = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="attributes.run_name LIKE 'AR_%'",
    output_format="pandas"
)

ar_score_df = pd.DataFrame({
    "unique_id": "UNRATE",
    "model_label": "AR",
    "model_name": "AR",
    "partition": runs.get(
        "params.partition",
        runs["tags.mlflow.runName"].str.replace("AR_", "", regex=False)
    ),
    "mae": pd.to_numeric(runs["metrics.mae"], errors="coerce"),
    "coverage": pd.to_numeric(runs.get("metrics.coverage"), errors="coerce"),
    "width": pd.to_numeric(runs.get("metrics.width"), errors="coerce"),
    "n": pd.to_numeric(runs.get("metrics.n"), errors="coerce"),
    "run_id": runs["run_id"],
})

client = MlflowClient()

def _n_from_artifact(run_id: str, partition: str):
    try:
        p = client.download_artifacts(run_id, "partition_summary.csv")
        df = pd.read_csv(p)

        # IMPORTANT: filtrer sur la bonne partition
        part = str(partition)
        row = df.loc[df["partition"].astype(str) == part]

        if len(row) == 1:
            return row["n_obs"].iloc[0]   # <-- ICI, pas df["n_obs"].iloc[0]
    except Exception:
        pass
    return None

mask = ar_score_df["n"].isna()
ar_score_df.loc[mask, "n"] = [
    _n_from_artifact(rid, part)
    for rid, part in zip(
        ar_score_df.loc[mask, "run_id"],
        ar_score_df.loc[mask, "partition"]
    )
]

# convertir n en entier (nullable)
ar_score_df["n"] = (
    pd.to_numeric(ar_score_df["n"], errors="coerce")
    .round()
    .astype("Int64")
)

# nettoyage final
ar_score_df["partition"] = ar_score_df["partition"].astype(str)
ar_score_df = (
    ar_score_df
    .drop(columns=["run_id"])
    .dropna(subset=["mae"])
    .sort_values("partition")
    .reset_index(drop=True)
)

print(ar_score_df)

  unique_id model_label model_name  partition       mae  coverage     width  \
0    UNRATE          AR         AR  1990-1999  0.494189  0.816667  1.763992   
1    UNRATE          AR         AR  2000-2008  0.514651  0.685185  1.569703   
2    UNRATE          AR         AR  2009-2019  0.763062  0.825758  3.086353   
3    UNRATE          AR         AR   2020-end  2.312401  0.661765  9.135620   
4    UNRATE          AR         AR        ALL  0.871151  0.761682  3.293990   

     n  
0  120  
1  108  
2  132  
3   68  
4  428  


## Leaderbord with exogenous models

In [30]:
# 1) Fusionner score_df (LR/RIDGE/LGBM) + ar_score_df (AR)
score_df_all = pd.concat([score_df, ar_score_df], ignore_index=True)

# (optionnel) s'assurer des bons types
score_df_all["partition"] = score_df_all["partition"].astype(str)
for c in ["mae", "coverage", "width"]:
    score_df_all[c] = pd.to_numeric(score_df_all[c], errors="coerce")
score_df_all["n"] = pd.to_numeric(score_df_all["n"], errors="coerce").astype("Int64")

# 2) Recalculer leaderboard (top 2 par partition, inclut AR)
leaderboard_all = (
    score_df_all.sort_values(
        by=["partition", "mae", "coverage", "width"],
        ascending=[True, True, False, True],
    )
    .groupby("partition", as_index=False)
    .head(2)
    .reset_index(drop=True)
)

print("=== SCORE (avec AR) ===")
print(score_df_all.sort_values(["partition", "mae"]).head(50))

print("\n=== TOP 2 par partition (avec AR) ===")
print(leaderboard_all)

=== SCORE (avec AR) ===
  unique_id model_label model_name  partition       mae  coverage     width  \
0    UNRATE          LR         LR  1990-1999  0.421220  0.800000  1.419842   
5    UNRATE          AR         AR  1990-1999  0.494189  0.816667  1.763992   
1    UNRATE          LR         LR  2000-2008  0.377789  0.648148  0.981396   
6    UNRATE          AR         AR  2000-2008  0.514651  0.685185  1.569703   
2    UNRATE          LR         LR  2009-2019  0.597483  0.780303  1.902029   
7    UNRATE          AR         AR  2009-2019  0.763062  0.825758  3.086353   
3    UNRATE          LR         LR   2020-end  1.776643  0.764706  6.250181   
8    UNRATE          AR         AR   2020-end  2.312401  0.661765  9.135620   
4    UNRATE          LR         LR        ALL  0.679970  0.750000  2.225355   
9    UNRATE          AR         AR        ALL  0.871151  0.761682  3.293990   

     n  
0  120  
5  120  
1  108  
6  108  
2  132  
7  132  
3   68  
8   68  
4  428  
9  428  

=== TO

# MLFLOW

Multivariate_experiment_design

In [37]:
# =====================================================
# MLflow logging complet (runs = model_label x partition)
# + Dataset name préfixé FEAST_ (ex: stationary_value:value)
# + Tags visibles dans l'UI (model_name, model_label, partition)
# ✅ Log "MLflow Model" (pour que la colonne "Models" ne soit plus "-")
# ❌ PAS de run "model catalog"
# =====================================================

import os
import io
import json
import joblib
import warnings
import logging
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.statsmodels
from contextlib import redirect_stdout, redirect_stderr

warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("H2_test")

# =====================================================
# 0) Data log (tous modèles)
# =====================================================
df_log = score_df_all.copy() if "score_df_all" in globals() else score_df.copy()

required_cols = {"model_label", "partition", "mae"}
missing = required_cols - set(df_log.columns)
if missing:
    raise ValueError(f"df_log missing required columns: {missing}")

df_log["model_label"] = df_log["model_label"].astype(str)
df_log["partition"]   = df_log["partition"].astype(str)
if "model_name" in df_log.columns:
    df_log["model_name"] = df_log["model_name"].astype(str)

# =====================================================
# 1) Résumé partitions (dataset input) + FEAST name
# =====================================================
if "bkt_score" not in globals():
    raise ValueError("bkt_score is missing in globals(). Needed for partition_summary_df.")

FEAST_FEATURE_NAME = "stationary_value:value"  # ✅ nom Feast voulu

def _format_feast_dataset_name(feast_name: str) -> str:
    # 'stationary_value:value' -> 'FEAST_stationary_value_value'
    clean = str(feast_name).replace(":", "_").replace("/", "_")
    return f"FEAST_{clean}"

DATASET_NAME = _format_feast_dataset_name(FEAST_FEATURE_NAME)

partition_summary_df = (
    bkt_score
    .groupby("partition", observed=True)
    .agg(
        n_obs=("ds", "size") if "ds" in bkt_score.columns else ("partition", "size"),
        ds_start=("ds", "min") if "ds" in bkt_score.columns else ("partition", "min"),
        ds_end=("ds", "max") if "ds" in bkt_score.columns else ("partition", "max"),
    )
    .reset_index()
)

dataset_obj = mlflow.data.from_pandas(
    partition_summary_df,
    source="feast",
    name=DATASET_NAME
)

# =====================================================
# 2) Helpers
# =====================================================
METRIC_SEQUENCE = "mae>coverage>width"

MODEL_PARAMS = {
    "AR": dict(
        horizon=12,
        min_train_n=36,
        trend="c",
        p_grid=list(range(1, 13)),
        cv_update_every_months=36,
        cv_anchor="1983-01-01",
        use_conformal=True,
        alpha=0.05,
        step_size=12,
        pi_windows=3,
        use_bagging=False
    ),
}

def _get_model_object(model_label: str, partition: str):
    """
    Supporte:
    - models_by_partition[partition] = model_obj
    - models_by_partition[model_label][partition] = model_obj
    """
    if "models_by_partition" not in globals():
        return None, "models_by_partition_missing"

    mbp = globals()["models_by_partition"]

    if isinstance(mbp, dict) and partition in mbp and not isinstance(mbp.get(partition), dict):
        return mbp.get(partition), "precomputed_partition"

    if isinstance(mbp, dict) and model_label in mbp and isinstance(mbp[model_label], dict):
        return mbp[model_label].get(partition), "precomputed_model_partition"

    return None, "not_found"

def _safe_float(x):
    try:
        return float(x)
    except Exception:
        return None

# =====================================================
# 3) Logging
# =====================================================
os.makedirs("artifacts_tmp", exist_ok=True)

_sink = io.StringIO()
with redirect_stdout(_sink), redirect_stderr(_sink):

    for _, row in df_log.iterrows():

        model_label = str(row["model_label"])
        model_name  = str(row.get("model_name", model_label))
        partition   = str(row["partition"])

        run_name = f"{model_label}_{partition}"

        with mlflow.start_run(run_name=run_name):

            # ✅ Tags visibles dans l'UI (et filtrables)
            mlflow.set_tag("mlflow.runName", run_name)
            mlflow.set_tag("model_label", model_label)
            mlflow.set_tag("model_name", model_name)
            mlflow.set_tag("partition", partition)
            mlflow.set_tag("metric_sequence", METRIC_SEQUENCE)

            # Dataset input (avec nom FEAST_...)
            mlflow.log_input(dataset_obj, context="evaluation")

            # Save dataset summary as artifact
            csv_path = "artifacts_tmp/partition_summary.csv"
            partition_summary_df.to_csv(csv_path, index=False)
            mlflow.log_artifact(csv_path, artifact_path="data_meta")

            # Params
            mlflow.log_param("model_label", model_label)
            mlflow.log_param("model_name", model_name)
            mlflow.log_param("partition", partition)
            mlflow.log_param("metric_sequence", METRIC_SEQUENCE)
            mlflow.log_param("dataset_name", DATASET_NAME)
            mlflow.log_param("feast_feature_name", FEAST_FEATURE_NAME)

            if model_label in MODEL_PARAMS:
                for k, v in MODEL_PARAMS[model_label].items():
                    mlflow.log_param(k, json.dumps(v) if isinstance(v, (list, dict)) else v)

            # Metrics
            mae_val = _safe_float(row.get("mae", None))
            if mae_val is not None:
                mlflow.log_metric("mae", mae_val)

            cov_val = _safe_float(row.get("coverage", None))
            if cov_val is not None and pd.notna(cov_val):
                mlflow.log_metric("coverage", cov_val)

            wid_val = _safe_float(row.get("width", None))
            if wid_val is not None and pd.notna(wid_val):
                mlflow.log_metric("width", wid_val)

            # Leaderboard (artifact)
            leaderboard_path = "artifacts_tmp/leaderboard.csv"
            df_log.to_csv(leaderboard_path, index=False)
            mlflow.log_artifact(leaderboard_path, artifact_path="leaderboard")

            # bkt_score artifacts
            if "bkt_score" in globals():
                os.makedirs("artifacts_tmp", exist_ok=True)

                # Par partition
                if "partition" in bkt_score.columns:
                    df_part = bkt_score[bkt_score["partition"].astype(str) == partition].copy()
                else:
                    df_part = bkt_score.copy()

                if len(df_part) > 0:
                    path_part = f"artifacts_tmp/bkt_{model_label}_{partition}.parquet"
                    df_part.to_parquet(path_part, index=False)
                    mlflow.log_artifact(path_part, artifact_path="data")

                # Full
                path_full = f"artifacts_tmp/bkt_score_full_{model_label}.parquet"
                cols_keep = [c for c in [
                    "ds",
                    "y_true",
                    "y_hat",
                    "lo_95",
                    "hi_95",
                    "partition",
                    "p_selected",
                ] if c in bkt_score.columns]

                if cols_keep:
                    bkt_score[cols_keep].to_parquet(path_full, index=False)
                    mlflow.log_artifact(path_full, artifact_path="data")

            # =====================================================
            # ✅ Model logging MLflow (pour remplir la colonne "Models")
            # =====================================================
            model_obj, model_source = _get_model_object(model_label, partition)

            if model_obj is not None:
                logged_as_mlflow_model = False

                # 1) Essai statsmodels
                try:
                    mlflow.statsmodels.log_model(model_obj, artifact_path="model")
                    logged_as_mlflow_model = True
                    mlflow.set_tag("model_flavor", "statsmodels")
                except Exception:
                    pass

                # 2) Essai sklearn
                if not logged_as_mlflow_model:
                    try:
                        mlflow.sklearn.log_model(model_obj, artifact_path="model")
                        logged_as_mlflow_model = True
                        mlflow.set_tag("model_flavor", "sklearn")
                    except Exception:
                        pass

                # 3) Fallback: artifact joblib si aucun flavor ne marche
                if not logged_as_mlflow_model:
                    model_path = f"artifacts_tmp/{model_label}_model_{partition}.joblib"
                    joblib.dump(model_obj, model_path)
                    mlflow.log_artifact(model_path, artifact_path="model")
                    mlflow.set_tag("model_flavor", "joblib_artifact_only")

                mlflow.log_param("model_logged", True)
                mlflow.log_param("model_source", model_source)
                mlflow.set_tag("model_logged", "true")
                mlflow.set_tag("model_source", model_source)

            else:
                mlflow.log_param("model_logged", False)
                mlflow.log_param("model_source", model_source)
                mlflow.set_tag("model_logged", "false")
                mlflow.set_tag("model_source", model_source)

print("Logging terminé")

Logging terminé
